In [1]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import math
import matplotlib.dates as mdates
from sklearn import datasets, linear_model
from mpl_toolkits.mplot3d import Axes3D
from matplotlib.lines import Line2D
import statsmodels.api as sm
from scipy.stats import t
from scipy.optimize import minimize
import os
import cvxpy as cp
from tqdm import tqdm
import seaborn as sns
import torch

import matplotlib.dates as mdates
import matplotlib.pyplot as plt
from scipy.stats import gaussian_kde
import openpyxl

# Data Preparation

In [3]:
data_path = "../data/processed/"

return_data = pd.read_csv(data_path + "stationary_return_data_subset.csv")

In [4]:
# Pivot return data
return_matrix = return_data.pivot(index='date', columns='RIC', values='return')
display(return_matrix)

RIC,BRO.N,CTRA.N,CVS.N,FFIV.OQ,HSY.N,LLY.N,NEM.N,PCG.N,REGN.OQ,TYL.N
date,,,,,,,,,,
2000-01-31,-0.104405,-0.081712,-1.236885e-01,-0.175439,-0.105263,0.005639,-0.168367,7.012195e-02,-0.034314,-0.204545
2000-02-29,-0.035086,0.074857,1.788909e-03,-0.042553,0.039651,-0.107465,0.085890,-5.982906e-02,3.588832,0.228571
2000-03-31,0.172348,0.142292,7.321429e-02,-0.247222,0.109531,0.059937,0.015535,3.281437e-02,-0.476770,0.104651
2000-04-28,0.037157,0.027682,1.595897e-01,-0.310886,-0.069231,0.227183,0.044568,2.351190e-01,-0.033827,-0.094737
2000-05-31,0.165747,0.345877,1.398903e-11,-0.309237,0.148974,-0.011941,-0.016000,1.444134e-11,-0.286652,-0.255814
...,...,...,...,...,...,...,...,...,...,...
2023-09-29,-0.057490,-0.040440,7.135185e-02,-0.015398,-0.068789,-0.030801,-0.052920,-1.042945e-02,-0.004271,-0.030846
2023-10-31,-0.004152,0.016636,-3.132471e-03,-0.059265,-0.063625,0.031277,0.014073,1.053937e-02,-0.052335,-0.034288
2023-11-30,0.076635,-0.038423,-1.536009e-02,0.129296,0.009148,0.068968,0.083216,5.337423e-02,0.056316,0.096380


In [6]:
# Create training and test data for the prediction model.

X = []
Y = []

max_lag = 3
n_rows = len(return_matrix)

n = max_lag
while n < n_rows:
    # Get X
    X_row = (return_matrix[n - max_lag:n]).values.flatten()
    X.append(X_row)

    # Get Y
    Y_row = (return_matrix.iloc[n]).values.flatten()
    Y.append(Y_row)
    n = n + 1

X = np.array(X)
Y = np.array(Y)

print(X.shape)
print(Y.shape)

(286, 30)
(286, 10)


In [ ]:
test_size = 12                          # 12 months (last year) for testing, rest for training

train_size = int(len(X) - test_size)

X_train = X[:train_size]
X_test = X[train_size:]

Y_train = Y[:train_size]
Y_test = Y[train_size:]

len(Y_train), len(Y_test)

(274, 12)

In [13]:
from sklearn.preprocessing import StandardScaler

x_scaler = StandardScaler()
y_scaler = StandardScaler()

X_train = x_scaler.fit_transform(X_train)
X_test = x_scaler.transform(X_test)

Y_train = y_scaler.fit_transform(Y_train)
Y_test = y_scaler.transform(Y_test)

In [14]:
X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
Y_train_tensor = torch.tensor(Y_train, dtype=torch.float32)

X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
Y_test_tensor = torch.tensor(Y_test, dtype=torch.float32)

In [15]:
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

batch_size = 4

train_dataset = TensorDataset(X_train_tensor, Y_train_tensor)
test_dataset = TensorDataset(X_test_tensor, Y_test_tensor)

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    shuffle=False
)

In [ ]:
class VARasNN(nn.Module):

    def __init__(self,input_dim,output_dim):

        super().__init__()

        self.linear = nn.Linear(
            input_dim,
            output_dim
        )

    def forward(self,x):

        return self.linear(x)

In [18]:
input_dim = X_train.shape[1]
output_dim = Y_train.shape[1]

model = VARasNN(input_dim, output_dim)

print(model)

VARasNN(
  (linear): Linear(in_features=30, out_features=10, bias=True)
)


In [19]:
criterion = nn.MSELoss()

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-4
)